# Causal Flow SQLite Build

This notebook flattens `runs.bson` and `runs_ablation_minimality.bson` into analysis-friendly SQLite tables. By default it includes ablation runs and marks them with ablation metadata.

In [45]:
import sqlite3
import importlib
import build_sqlite
importlib.reload(build_sqlite)

from build_sqlite import DB_PATH, load_database, table_counts

In [46]:
# %load_ext autoreload
# %autoreload 2

In [47]:
conn, load_summary = load_database(include_ablations=True)

## Tables

- `runs`: one row per experiment run
- `traces`: one row per problem attempt, passing or failing
- `steps`: one row per agent trace step
- `repair_attempts`: one row per successful repairable step currently recoverable from the BSON
- `judge_votes`: one row per individual LLM judge vote
- `consensus_steps`: one row per step accepted by the multi-agent critique/council
- `trace_metrics`: one row per failed trace with attribution, repair, minimality, and critique summary metrics
- `triage_labels`: one row per failed trace for the Trace Triage paper labels; CausalFlow-repaired traces are auto-labeled `LOCAL_REPAIR`

In [48]:
table_counts(conn)

{'runs': 24,
 'traces': 3893,
 'steps': 41615,
 'repair_attempts': 2522,
 'judge_votes': 2624,
 'consensus_steps': 2512,
 'trace_metrics': 1739,
 'triage_labels': 1739}

In [27]:
for row in conn.execute("""
    SELECT benchmark,
           COUNT(*) AS traces,
           SUM(success = 1) AS passing,
           SUM(success = 0) AS failing,
           ROUND(AVG(success), 3) AS trace_accuracy
    FROM traces
    GROUP BY benchmark
    ORDER BY benchmark
"""):
    print(row)

In [28]:
for row in conn.execute("""
    SELECT judge_says_causal,
           is_repairable_step,
           COUNT(*) AS votes
    FROM judge_votes
    GROUP BY judge_says_causal, is_repairable_step
    ORDER BY judge_says_causal, is_repairable_step
"""):
    print(row)

In [ ]:
# Use this later if the database already exists and you do not want to rebuild it.
# conn = sqlite3.connect(DB_PATH)